# 02 Prepare Labeled ML Dataset

Purpose: transform raw PR suggestion pairs into ML-ready files, review examples, and supervised-training rows.

This notebook does not call GitHub. It consumes raw local data from `data/raw/...`.

Safety rule: rebuilds write to `dataset_candidate/` first, so the existing hand-labeled `dataset/labels.csv` is not overwritten.


In [ ]:
from __future__ import annotations

import csv
import json
import os
import subprocess
from collections import Counter
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
RAW_PAIRS_JSONL = PROJECT_ROOT / "data" / "raw" / "pipeline-fl-control-plane-closed-prs" / "export" / "raw_pairs.jsonl"
CURRENT_DATASET_DIR = PROJECT_ROOT / "data" / "processed" / "pr_suggestion_coverage" / "dataset"
CANDIDATE_DATASET_DIR = PROJECT_ROOT / "data" / "processed" / "pr_suggestion_coverage" / "dataset_candidate"
ACTIVE_DATASET_DIR = CURRENT_DATASET_DIR
REVIEW_EXAMPLE_LIMIT = None  # set to an integer for a smaller manual-review batch

for path in [RAW_PAIRS_JSONL.parent, CURRENT_DATASET_DIR, CANDIDATE_DATASET_DIR]:
    path.mkdir(parents=True, exist_ok=True)

RAW_PAIRS_JSONL, ACTIVE_DATASET_DIR, CANDIDATE_DATASET_DIR


## Build Candidate Dataset Files

This converts raw pairs into a candidate dataset.

It writes to `dataset_candidate/`, not the active labeled `dataset/`, because `src/pr_suggestion_metrics/build_dataset.py` rewrites `labels.csv`.

After reviewing the candidate output, manually promote it to `dataset/` if that is intentional.


In [ ]:
if not RAW_PAIRS_JSONL.exists():
    raise FileNotFoundError(f"Raw pairs file does not exist: {RAW_PAIRS_JSONL}")

cmd = [
    "uv",
    "run",
    "--locked",
    "pr-suggestion-build-dataset",
    str(RAW_PAIRS_JSONL),
    "--output-dir",
    str(CANDIDATE_DATASET_DIR),
]

env = os.environ.copy()
env.setdefault("UV_CACHE_DIR", "/private/tmp/uv-cache")
print("Reading raw pairs:", RAW_PAIRS_JSONL)
print("Writing candidate dataset:", CANDIDATE_DATASET_DIR)
subprocess.run(cmd, cwd=PROJECT_ROOT, env=env, check=True)


## Inspect Active Labels

This reads `ACTIVE_DATASET_DIR`, which defaults to the current hand-labeled dataset.


In [ ]:
labels_path = ACTIVE_DATASET_DIR / "labels.csv"
dataset_path = ACTIVE_DATASET_DIR / "dataset.jsonl"

with labels_path.open(newline="", encoding="utf-8") as labels_file:
    labels = list(csv.DictReader(labels_file))

print("label rows:", len(labels))
print("label counts:", Counter(row["label"] for row in labels))
print("suggested label counts:", Counter(row["suggested_label"] for row in labels))

[
    {
        "example_id": row["example_id"],
        "label": row["label"],
        "suggested_label": row["suggested_label"],
        "suggested_percentage": row["suggested_percentage"],
        "pr_url": row["pr_url"],
    }
    for row in labels[:20]
]

## Create Review Examples

In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open(encoding="utf-8") as stream:
        for line in stream:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

review_dir = ACTIVE_DATASET_DIR / "review_examples"
review_dir.mkdir(parents=True, exist_ok=True)

dataset_rows = read_jsonl(dataset_path)
rows_for_review = dataset_rows if REVIEW_EXAMPLE_LIMIT is None else dataset_rows[:REVIEW_EXAMPLE_LIMIT]
for row in rows_for_review:
    review_path = review_dir / f"{row['example_id']}.md"
    review_path.write_text(
        "\n".join([
            f"# {row['example_id']}",
            "",
            f"PR: {row['pr_url']}",
            f"Suggested label: {row['deterministic_landed_estimate']}%",
            f"File overlap: {row['file_overlap_ratio']}",
            f"Changed-line overlap: {row['changed_line_overlap_ratio']}",
            "",
            "## Suggested diff",
            "```diff",
            row["suggested_diff"],
            "```",
            "",
            "## Landed PR diff",
            "```diff",
            row["landed_diff"],
            "```",
            "",
        ]),
        encoding="utf-8",
    )

print("review files:", review_dir)
print("review file count:", len(list(review_dir.glob("*.md"))))

## Create Supervised Training JSONL

Rows without a final human label are skipped.

In [ ]:
training_jsonl = ACTIVE_DATASET_DIR / "supervised_training.jsonl"
with labels_path.open(newline="", encoding="utf-8") as labels_file:
    labels_by_id = {row["example_id"]: row for row in csv.DictReader(labels_file)}

written = 0
with training_jsonl.open("w", encoding="utf-8") as stream:
    for row in read_jsonl(dataset_path):
        label_row = labels_by_id.get(row["example_id"], {})
        label = str(label_row.get("label") or "").strip()
        if label not in {"0%", "partial", "mostly", "100%"}:
            continue
        record = {
            "id": row["example_id"],
            "input": {
                "suggested_diff": row["suggested_diff"],
                "landed_diff": row["landed_diff"],
                "repo": row["repo"],
                "pr_url": row["pr_url"],
                "suggested_stats": row["suggested_stats"],
                "landed_stats": row["landed_stats"],
            },
            "target": {
                "label": label,
                "expected_landed_percentage": label_row.get("expected_landed_percentage"),
                "notes": label_row.get("label_notes"),
            },
        }
        stream.write(json.dumps(record, ensure_ascii=False) + "\n")
        written += 1

print("wrote", written, "training rows to", training_jsonl)